# Fidelity and Utility Evaluation Demo

This notebook evaluates synthetic tabular data using the current project data layout.


In [ ]:
import pandas as pd
import os
import numpy as np

# config

In [ ]:
from pathlib import Path

DEMO_DIR = Path.cwd().resolve()
if DEMO_DIR.name != "demo":
    DEMO_DIR = DEMO_DIR / "demo"

# Set this to the baseline model you want to evaluate.
curr_model = "CTGAN"
seeds = [42]

real_train_root = DEMO_DIR / "origin_train_dataset"
real_test_root = DEMO_DIR / "origin_test_dataset"
baseline_syn_root = DEMO_DIR / "baseline_data"
enhanced_syn_root = DEMO_DIR / "enhanced_data"
report_dir = DEMO_DIR / "reports"

data_names = [
    "adult",
    "bank",
    "Customertravel",
    "german_credit",
    "insurance",
    "StudentsPerformance",
    "Telco-Customer-Churn",
    "sick",
    "Obesity",
]

backbones = {
    "gpt2": "gpt2",
    # "bert": "bert-base-uncased",
    # "ModernBERT": "ModernBERT-base",
    # "qwen2": "Qwen2-7B",
    # "llama3": "Meta-Llama-3-8B",
    # "bge": "bge-large-en-v1.5"
}

TARGET_COL = {
    "adult": "class",
    "bank": "deposit",
    "Customertravel": "Target",
    "german_credit": "class",
    "insurance": "smoker",
    "StudentsPerformance": "test preparation course",
    "Telco-Customer-Churn": "Churn",
    "sick": "Class",
    "Obesity": "NObeyesdad",
}


# utils

In [ ]:
def generate_metadata(df: pd.DataFrame):
    """Build SDMetrics metadata from a dataframe."""
    metadata = {"columns": {}, "primary_key": None}
    for col in df.columns:
        dtype = str(df[col].dtype)
        if dtype.startswith(("int", "float")):
            sdtype = "numerical"
        elif dtype == "bool":
            sdtype = "boolean"
        elif "datetime" in dtype:
            sdtype = "datetime"
        else:
            sdtype = "categorical"
        metadata["columns"][col] = {"sdtype": sdtype}
    return metadata


def save_aggregated_report(results_list, groupby_cols, metric_cols, save_path, raw_save_path=None):
    """Save raw metric rows and a mean/std summary."""
    df = pd.DataFrame(results_list)
    save_path = Path(save_path)
    raw_save_path = Path(raw_save_path) if raw_save_path else save_path.with_name(save_path.name.replace("_Summary.csv", "_Raw.csv"))

    save_path.parent.mkdir(parents=True, exist_ok=True)
    raw_save_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(raw_save_path, index=False)

    agg_dict = {col: ["mean", "std"] for col in metric_cols}
    summary_df = df.groupby(groupby_cols).agg(agg_dict).reset_index()
    summary_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary_df.columns]
    summary_df.to_csv(save_path, index=False)
    return summary_df


# Load Data

This demo reads real training tables from `demo/origin_train_dataset`, real test tables from `demo/origin_test_dataset`, baseline synthetic tables from `demo/baseline_data`, and enhanced synthetic tables from `demo/enhanced_data`. Fidelity uses the corresponding real train split; utility uses the corresponding real test split.


In [ ]:
def read_table(path, drop_runtime_cols=True):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")

    df = pd.read_csv(path)
    if drop_runtime_cols:
        df = df.drop(columns=["__row_id__", "text", ""], axis=1, errors="ignore")
    else:
        df = df.drop(columns=[""], axis=1, errors="ignore")
    return df


real_train_dfs = {seed: {} for seed in seeds}
real_test_dfs = {seed: {} for seed in seeds}
metadatas = {seed: {} for seed in seeds}

for seed in seeds:
    for dn in data_names:
        train_path = real_train_root / f"seed_{seed}" / f"{dn}.csv"
        test_path = real_test_root / f"seed_{seed}" / f"{dn}.csv"

        train_df = read_table(train_path, drop_runtime_cols=True)
        test_df = read_table(test_path, drop_runtime_cols=True)

        real_train_dfs[seed][dn] = train_df
        real_test_dfs[seed][dn] = test_df
        metadatas[seed][dn] = generate_metadata(train_df)


syn_dfs = {seed: {dn: {} for dn in data_names} for seed in seeds}

for seed in seeds:
    for dn in data_names:
        baseline_path = (
            baseline_syn_root / curr_model / "syn_data" /
            f"seed_{seed}" / dn / f"syn_{dn}.csv"
        )
        syn_dfs[seed][dn][curr_model] = read_table(baseline_path, drop_runtime_cols=True)

        for bb_key, bb_path in backbones.items():
            enhanced_path = (
                enhanced_syn_root / f"seed_{seed}" / bb_path /
                curr_model / dn / f"syn_{dn}.csv"
            )
            model_name = f"{curr_model} + {bb_key}"
            syn_dfs[seed][dn][model_name] = read_table(enhanced_path, drop_runtime_cols=True)

print("[DONE] Data loaded.")


# Fidelity

Fidelity metrics compare each synthetic table with the corresponding real training split in `demo/origin_train_dataset`.


## QualityReport
`Column Shapes measures per-column distribution similarity. Column Pair Trends measures whether pairwise relationships are preserved.`


In [ ]:
from sdmetrics.reports.single_table import QualityReport

rows = []

for seed in seeds:
    for dn in data_names:
        real_df = real_train_dfs[seed][dn]
        meta = metadatas[seed][dn]
        
        for model_name, syn_df in syn_dfs[seed][dn].items():
            report = QualityReport()
            report.generate(real_df, syn_df, meta)

            props = report.get_properties()
            col_shapes = props.loc[props["Property"] == "Column Shapes", "Score"].values[0]
            col_pairs = props.loc[props["Property"] == "Column Pair Trends", "Score"].values[0]
            overall = report.get_score()
            
            rows.append({
                "Dataset": dn,
                "Model": model_name,
                "Seed": seed,
                "Column Shapes": col_shapes,
                "Column Pair Trends": col_pairs,
                "Overall Quality Score": overall,
            })

save_path = report_dir / "fidelity" / curr_model / "QualityReport_Summary.csv"
quality_summary = save_aggregated_report(
    rows, 
    groupby_cols=["Dataset", "Model"], 
    metric_cols=["Column Shapes", "Column Pair Trends", "Overall Quality Score"], 
    save_path=save_path
)

display(quality_summary.head())


## DiagnosticReport
`Data Validity checks invalid values or categories. Data Structure checks whether column names and data types match the real table.`


In [ ]:
from sdmetrics.reports.single_table import DiagnosticReport

rows = []

for seed in seeds:
    for dn in data_names:
        real_df = real_train_dfs[seed][dn]
        meta = metadatas[seed][dn]
        
        for model_name, syn_df in syn_dfs[seed][dn].items():
            report = DiagnosticReport()
            report.generate(real_df, syn_df, meta)

            props = report.get_properties()
            data_validity = props.loc[props["Property"] == "Data Validity", "Score"].values[0]
            data_structure = props.loc[props["Property"] == "Data Structure", "Score"].values[0]

            overall = report.get_score()
            
            rows.append({
                "Dataset": dn,
                "Model": model_name,
                "Seed": seed,
                "Data Validity": data_validity,
                "Data Structure": data_structure,
                "Overall Diagnostic Score": overall,
            })

save_path = report_dir / "fidelity" / curr_model / "DiagnosticReport_Summary.csv"
quality_summary = save_aggregated_report(
    rows, 
    groupby_cols=["Dataset", "Model"], 
    metric_cols=["Data Validity", "Data Structure", "Overall Diagnostic Score"], 
    save_path=save_path
)

display(quality_summary.head())


# Utility

Utility metrics use each seed's real train split as the upper-bound training data and the matching real test split as the evaluation data. TSTR trains downstream classifiers on synthetic data and evaluates them on the matching real test split.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone

rows = []
f1_type = "macro"

# Define downstream classifiers.
# random_state is reset for each seed when the classifier supports it.
test_classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric="logloss"),
}


def make_preprocess(X_train_real):
    """Fit preprocessing on the real training split only."""
    num_cols = X_train_real.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns
    cat_cols = X_train_real.select_dtypes(include=["object", "category", "bool"]).columns

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop"
    )
    preprocess.fit(X_train_real)
    return preprocess


def align_predict_proba(clf, X_test_enc, global_classes):
    """Align predict_proba columns to the full real-train label space."""
    proba = clf.predict_proba(X_test_enc)
    aligned = np.zeros((len(X_test_enc), len(global_classes)), dtype=float)
    class_to_col = {label: idx for idx, label in enumerate(global_classes)}

    for local_idx, label in enumerate(clf.classes_):
        if label in class_to_col:
            aligned[:, class_to_col[label]] = proba[:, local_idx]

    return aligned


def safe_auc_score(y_true, y_proba, global_classes):
    """Return macro AUC when defined; otherwise NaN for degenerate splits."""
    try:
        if len(np.unique(y_true)) < 2 or y_proba is None:
            return np.nan
        if len(global_classes) == 2:
            return roc_auc_score(y_true, y_proba[:, 1])
        return roc_auc_score(
            y_true,
            y_proba,
            labels=global_classes,
            multi_class="ovr",
            average="macro",
        )
    except ValueError:
        return np.nan


def fit_predict_downstream(clf_name, clf_obj, X_train_enc, y_train_global, X_test_enc, seed, global_classes):
    """Fit downstream classifier and return predictions plus aligned probabilities."""
    unique_labels = np.unique(y_train_global)

    # If synthetic target has one class, use a constant dummy classifier.
    if len(unique_labels) == 1:
        dummy = DummyClassifier(strategy="constant", constant=unique_labels[0])
        dummy.fit(X_train_enc, y_train_global)
        y_pred = dummy.predict(X_test_enc)
        y_proba = align_predict_proba(dummy, X_test_enc, global_classes)
        return y_pred, y_proba

    clf = clone(clf_obj)
    if hasattr(clf, "random_state"):
        clf.set_params(random_state=seed)

    # XGBoost expects contiguous local labels during training.
    if clf_name == "XGBoost":
        local_le = LabelEncoder()
        y_train_local = local_le.fit_transform(y_train_global)
        clf.fit(X_train_enc, y_train_local)
        y_pred_local = clf.predict(X_test_enc)
        y_pred = local_le.inverse_transform(y_pred_local.astype(int))
        local_proba = clf.predict_proba(X_test_enc)
        y_proba = np.zeros((len(X_test_enc), len(global_classes)), dtype=float)
        class_to_col = {label: idx for idx, label in enumerate(global_classes)}
        for local_idx, label in enumerate(local_le.classes_):
            if label in class_to_col:
                y_proba[:, class_to_col[label]] = local_proba[:, local_idx]
        return y_pred, y_proba

    clf.fit(X_train_enc, y_train_global)
    y_pred = clf.predict(X_test_enc)
    y_proba = align_predict_proba(clf, X_test_enc, global_classes)
    return y_pred, y_proba


for seed in seeds:
    print(f"[INFO] Processing seed: {seed}")

    for dn in data_names:
        print(f"[INFO] Processing dataset: {dn}")
        target_col = TARGET_COL[dn]

        real_train_df = real_train_dfs[seed][dn].copy()
        real_test_df = real_test_dfs[seed][dn].copy()

        if target_col not in real_train_df.columns:
            raise KeyError(f"Target column '{target_col}' not found in real_train_df for dataset {dn}, seed {seed}")
        if target_col not in real_test_df.columns:
            raise KeyError(f"Target column '{target_col}' not found in real_test_df for dataset {dn}, seed {seed}")

        # Fit label encoding on the real training label space.
        le = LabelEncoder()
        y_train_real = le.fit_transform(real_train_df[target_col])
        valid_labels = set(le.classes_)
        global_classes = np.arange(len(le.classes_))

        # Drop test rows with labels unseen in the real training split.
        test_valid_mask = real_test_df[target_col].isin(valid_labels)
        test_valid_ratio = test_valid_mask.mean()
        if test_valid_ratio < 1.0:
            print(f"[WARN] Test set contains unseen labels: dataset={dn}, seed={seed}, valid_ratio={test_valid_ratio:.4f}")
            real_test_df = real_test_df[test_valid_mask].copy()

        X_train_real = real_train_df.drop(columns=[target_col])
        X_test_real = real_test_df.drop(columns=[target_col])
        y_test_real = le.transform(real_test_df[target_col])

        preprocess = make_preprocess(X_train_real)
        X_train_real_enc = preprocess.transform(X_train_real)
        X_test_real_enc = preprocess.transform(X_test_real)

        # Real upper bound: train on real train, evaluate on real test.
        for clf_name, clf_obj in test_classifiers.items():
            print(f"[INFO] Evaluating real upper bound: classifier={clf_name}")

            y_pred, y_proba = fit_predict_downstream(
                clf_name=clf_name,
                clf_obj=clf_obj,
                X_train_enc=X_train_real_enc,
                y_train_global=y_train_real,
                X_test_enc=X_test_real_enc,
                seed=seed,
                global_classes=global_classes,
            )

            real_acc = accuracy_score(y_test_real, y_pred)
            real_f1 = f1_score(y_test_real, y_pred, average=f1_type, zero_division=0)
            real_auc = safe_auc_score(y_test_real, y_proba, global_classes)

            rows.append({
                "Dataset": dn,
                "Model": "Real (Upper Bound)",
                "Seed": seed,
                "Classifier": clf_name,
                "Accuracy": real_acc,
                "F1": real_f1,
                "AUC": real_auc,
                "Valid_Ratio": 1.0,
                "Test_Valid_Ratio": test_valid_ratio,
                "Overall Utility": (real_acc + real_f1) / 2,
            })

        # TSTR: train on synthetic data, evaluate on real test.
        for model_name, syn_df in syn_dfs[seed][dn].items():
            print(f"[INFO] Evaluating synthetic model: {model_name}")

            if target_col not in syn_df.columns:
                raise KeyError(f"Target column '{target_col}' not found in synthetic data for model {model_name}, dataset {dn}, seed {seed}")

            # Keep synthetic rows whose target labels exist in the real training split.
            is_valid_mask = syn_df[target_col].isin(valid_labels)
            valid_ratio = is_valid_mask.mean()
            syn_df_clean = syn_df[is_valid_mask].copy()

            if valid_ratio < 1.0:
                print(f"[WARN] Invalid synthetic target labels: model={model_name}, seed={seed}, valid_ratio={valid_ratio:.4f}")

            if len(syn_df_clean) == 0:
                print(f"[WARN] Skipping synthetic model with no valid labels: model={model_name}, seed={seed}")
                continue

            X_syn = syn_df_clean.drop(columns=[target_col])
            y_syn = le.transform(syn_df_clean[target_col])

            # Align synthetic feature columns to the real training columns.
            X_syn = X_syn.reindex(columns=X_train_real.columns)
            X_syn_enc = preprocess.transform(X_syn)

            for clf_name, clf_obj in test_classifiers.items():
                print(f"[INFO] Running TSTR: classifier={clf_name}")

                y_pred, y_proba = fit_predict_downstream(
                    clf_name=clf_name,
                    clf_obj=clf_obj,
                    X_train_enc=X_syn_enc,
                    y_train_global=y_syn,
                    X_test_enc=X_test_real_enc,
                    seed=seed,
                    global_classes=global_classes,
                )

                acc = accuracy_score(y_test_real, y_pred)
                f1 = f1_score(y_test_real, y_pred, average=f1_type, zero_division=0)
                auc = safe_auc_score(y_test_real, y_proba, global_classes)

                rows.append({
                    "Dataset": dn,
                    "Model": model_name,
                    "Seed": seed,
                    "Classifier": clf_name,
                    "Accuracy": acc,
                    "F1": f1,
                    "AUC": auc,
                    "Valid_Ratio": valid_ratio,
                    "Test_Valid_Ratio": test_valid_ratio,
                    "Overall Utility": (acc + f1) / 2,
                })

# Save report
save_path = report_dir / "utility" / curr_model / "MLUtility_TSTR_Summary.csv"

utility_summary = save_aggregated_report(
    rows,
    groupby_cols=["Dataset", "Model", "Classifier"],
    metric_cols=["Accuracy", "F1", "AUC", "Valid_Ratio", "Test_Valid_Ratio", "Overall Utility"],
    save_path=save_path,
)

print(f"[DONE] Utility report saved: {save_path}")
display(utility_summary.head())
